# 02 - Data Understanding

This notebook inspects the raw household power consumption dataset before cleaning or modeling. The goal is to understand the columns, actual local date range, missing values, and first consumption patterns.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "DataSet" / "household_power_consumption.csv"
DATA_PATH

## Load Raw Data

The raw CSV contains date and time columns plus power and sub-metering measurements. Missing values may appear as `?`, so they are registered as missing during loading.

In [ ]:
raw_df = pd.read_csv(DATA_PATH, na_values=["?"], low_memory=False)
print("Raw shape:", raw_df.shape)
display(raw_df.head())

In [ ]:
raw_df.info()

## Actual Local Date Range

The public source dataset may cover a longer period, but the project must report the date range that exists in the local CSV being analyzed.

In [ ]:
datetime_series = pd.to_datetime(
    raw_df["Date"].astype(str) + " " + raw_df["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce",
)

print("Start:", datetime_series.min())
print("End:", datetime_series.max())
print("Rows:", len(raw_df))

## Missing Values

Missing values are important because time-series models and clustering algorithms cannot work correctly with raw missing entries.

In [ ]:
missing_summary = raw_df.isna().sum().sort_values(ascending=False)
display(missing_summary.to_frame("missing_count"))

In [ ]:
numeric_columns = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]

raw_numeric = raw_df[numeric_columns].apply(pd.to_numeric, errors="coerce")
display(raw_numeric.describe().T)

## First Visual Pattern

A small sample is plotted first to verify that the data behaves like an ordered time series before deeper preparation.

In [ ]:
sample_df = raw_df.head(5000).copy()
sample_df["datetime"] = pd.to_datetime(
    sample_df["Date"].astype(str) + " " + sample_df["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce",
)
sample_df["Global_active_power"] = pd.to_numeric(sample_df["Global_active_power"], errors="coerce")

plt.figure(figsize=(12, 4))
plt.plot(sample_df["datetime"], sample_df["Global_active_power"])
plt.title("Sample Global Active Power Over Time")
plt.xlabel("Datetime")
plt.ylabel("Global active power (kW)")
plt.tight_layout()
plt.show()